In [3]:
import pandas as pd

# Cara 1: Menggunakan Dictionary (Kolom sebagai Key)
data_dict = {
    "id_transaksi": [101, 102, 103, 104],
    "kota": ["Magelang", "Yogyakarta", "Magelang", "Semarang"],
    "kategori": ["Elektronik", "Pakaian", "Makanan", "Elektronik"],
    "jumlah": [2, 5, 10, 1],
    "harga": [1500000, 45000, 15000, 3000000]
}
df_dict = pd.DataFrame(data_dict)

# Cara 2: Menggunakan List of Lists (Data per Baris)
data_baris = [
    [101, "Magelang", "Elektronik", 2, 1500000],
    [102, "Yogyakarta", "Pakaian", 5, 45000],
    [103, "Magelang", "Makanan", 10, 15000],
    [104, "Semarang", "Elektronik", 1, 3000000]
]
kolom = ["id_transaksi", "kota", "kategori", "jumlah", "harga"]
df_baris = pd.DataFrame(data=data_baris, columns=kolom)

df_dict.describe()


,id_transaksi,jumlah,harga
count,4.000000,4.000000,4.000000e+00
mean,102.500000,4.500000,1.140000e+06
std,1.290994,4.041452,1.420546e+06
min,101.000000,1.000000,1.500000e+04
25%,101.750000,1.750000,3.750000e+04
50%,102.500000,3.500000,7.725000e+05
75%,103.250000,6.250000,1.875000e+06
max,104.000000,10.000000,3.000000e+06


In [11]:
# 1. Menambah kolom baru
df_dict["total_harga"] = df_dict["jumlah"] * df_dict["harga"]

# 2. Filtering Data (Multi-Kondisi)
# Ambil transaksi di Magelang DENGAN total_harga > 100.000
df_filter = df_dict[(df_dict["kota"] == "Magelang") & (df_dict["total_harga"] > 100000)]

# 3. Agregasi Data (GroupBy)
# Menghitung total pendapatan per kota
df_group = df_dict.groupby("kota")["total_harga"].sum().reset_index()
df_filter

,id_transaksi,kota,kategori,jumlah,harga,total_harga
0,101,Magelang,Elektronik,2,1500000,3000000
2,103,Magelang,Makanan,10,15000,150000


In [21]:
# --- PANDAS ---
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("JoinExample").getOrCreate()


df_mhs = pd.DataFrame({"id_mhs": [1, 2, 3], "nama": ["Budi", "Siti", "Andi"]})
df_ukm = pd.DataFrame({"id_mhs": [2, 3, 4], "ukm": ["Basket", "Futsal", "Musik"]})

# Left Join
df_left = pd.merge(df_mhs, df_ukm, on="id_mhs", how="left")

sp_mhs = spark.createDataFrame(df_mhs)
sp_ukm = spark.createDataFrame(df_ukm)

# Inner Join di PySpark
sp_inner = sp_mhs.join(sp_ukm, on="id_mhs", how="inner")

sp_inner

DataFrame[id_mhs: bigint, nama: string, ukm: string]

In [18]:
# --- PYSPARK ---
from pyspark.sql import SparkSession

sp_mhs = spark.createDataFrame(df_mhs)
sp_ukm = spark.createDataFrame(df_ukm)

# Inner Join di PySpark
sp_inner = sp_mhs.join(sp_ukm, on="id_mhs", how="inner")

In [25]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("ModulPySpark").getOrCreate()

# Membuat PySpark DataFrame dari Data Dictionary
data_sp = [
    (1, "Laptop", "Elektronik", 2, 7000000),
    (2, "Kaos", "Pakaian", 10, 50000),
    (3, "HP", "Elektronik", 1, 3500000),
    (4, "Kemeja", "Pakaian", 4, 120000)
]
kolom_sp = ["id", "produk", "kategori", "jumlah", "harga_satuan"]

df_spark = spark.createDataFrame(data_sp, schema=kolom_sp)
data_sp

[(1, 'Laptop', 'Elektronik', 2, 7000000),
 (2, 'Kaos', 'Pakaian', 10, 50000),
 (3, 'HP', 'Elektronik', 1, 3500000),
 (4, 'Kemeja', 'Pakaian', 4, 120000)]

In [23]:
spark.stop()

In [26]:
# 1. Menambah Kolom Baru (withColumn + F.col)
df_spark = df_spark.withColumn("total_pembelian", F.col("jumlah") * F.col("harga_satuan"))

# 2. Memfilter Data (filter + F.col)
# Ambil produk kategori Elektronik atau yang harganya di atas 100.000
df_elektronik = df_spark.filter((F.col("kategori") == "Elektronik") | (F.col("harga_satuan") > 100000))

# 3. Pengelompokan & Agregasi (groupBy + agg)
df_analisis = df_spark.groupBy("kategori").agg(
    F.sum("total_pembelian").alias("total_pendapatan"),
    F.avg("harga_satuan").alias("rata_rata_harga")
)

# 4. Tampilkan Hasil (di PySpark wajib panggil .show())
df_analisis.show()

+----------+----------------+---------------+
|  kategori|total_pendapatan|rata_rata_harga|
+----------+----------------+---------------+
|Elektronik|        17500000|      5250000.0|
|   Pakaian|          980000|        85000.0|
+----------+----------------+---------------+



In [27]:
import pandas as pd

# 1. Dataset Penjualan
data_penjualan = {
    "id_transaksi": [101, 102, 103, 104, 105, 106],
    "id_pelanggan": [1, 2, 1, 3, 2, 4],
    "kategori": ["Elektronik", "Pakaian", "Elektronik", "Makanan", "Pakaian", "Makanan"],
    "jumlah": [2, 5, 1, 10, 3, 8],
    "harga_satuan": [1500000, 50000, 3000000, 15000, 75000, 20000]
}
df_penjualan = pd.DataFrame(data_penjualan)

# 2. Dataset Pelanggan
data_pelanggan = {
    "id_pelanggan": [1, 2, 3, 4],
    "nama_pelanggan": ["Budi", "Siti", "Andi", "Dewi"],
    "kota": ["Magelang", "Yogyakarta", "Magelang", "Semarang"]
}
df_pelanggan = pd.DataFrame(data_pelanggan)

# Menampilkan kedua DataFrame
display(df_penjualan)
display(df_pelanggan)

,id_transaksi,id_pelanggan,kategori,jumlah,harga_satuan
0,101,1,Elektronik,2,1500000
1,102,2,Pakaian,5,50000
2,103,1,Elektronik,1,3000000
3,104,3,Makanan,10,15000
4,105,2,Pakaian,3,75000
5,106,4,Makanan,8,20000


,id_pelanggan,nama_pelanggan,kota
0,1,Budi,Magelang
1,2,Siti,Yogyakarta
2,3,Andi,Magelang
3,4,Dewi,Semarang


In [28]:
import pandas as pd

# 1. Dataset Penjualan
data_penjualan = {
    "id_transaksi": [101, 102, 103, 104, 105, 106],
    "id_pelanggan": [1, 2, 1, 3, 2, 4],
    "kategori": ["Elektronik", "Pakaian", "Elektronik", "Makanan", "Pakaian", "Makanan"],
    "jumlah": [2, 5, 1, 10, 3, 8],
    "harga_satuan": [1500000, 50000, 3000000, 15000, 75000, 20000]
}
df_penjualan = pd.DataFrame(data_penjualan)

# 2. Dataset Pelanggan
data_pelanggan = {
    "id_pelanggan": [1, 2, 3, 4],
    "nama_pelanggan": ["Budi", "Siti", "Andi", "Dewi"],
    "kota": ["Magelang", "Yogyakarta", "Magelang", "Semarang"]
}
df_pelanggan = pd.DataFrame(data_pelanggan)

# Menampilkan kedua DataFrame
display(df_penjualan)
display(df_pelanggan)

,id_transaksi,id_pelanggan,kategori,jumlah,harga_satuan
0,101,1,Elektronik,2,1500000
1,102,2,Pakaian,5,50000
2,103,1,Elektronik,1,3000000
3,104,3,Makanan,10,15000
4,105,2,Pakaian,3,75000
5,106,4,Makanan,8,20000


,id_pelanggan,nama_pelanggan,kota
0,1,Budi,Magelang
1,2,Siti,Yogyakarta
2,3,Andi,Magelang
3,4,Dewi,Semarang


Hitung Kolom Baru: Tambahkan kolom total_harga di df_penjualan (jumlah * harga_satuan).

JOIN Tabel (Pandas): Penggabungan LEFT JOIN antara df_penjualan dan df_pelanggan pada kolom id_pelanggan (simpan sebagai df_merged).

Konversi & Filter (PySpark): Ubah df_merged ke PySpark DataFrame, lalu filter transaksi dari kota Magelang dengan total_harga > 100000.

Agregasi (PySpark): Hitung total pendapatan dan rata-rata jumlah barang terjual per kategori.

In [64]:
#df_penjualan["total_harga"] = df_penjualan["jumlah"] * df_penjualan["harga_satuan"]
#df_penjualan

df_merged = pd.merge(df_penjualan, df_pelanggan, on="id_pelanggan", how= "left")
df_merged

df_filter = df_merged[(df_merged["kota"] == "Magelang") & (df_merged["total_harga"] > 100000)]
df_filter

,id_transaksi,id_pelanggan,kategori,jumlah,harga_satuan,total_harga,nama_pelanggan,kota
0,101,1,Elektronik,2,1500000,3000000,Budi,Magelang
2,103,1,Elektronik,1,3000000,3000000,Budi,Magelang
3,104,3,Makanan,10,15000,150000,Andi,Magelang


In [69]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Latihan_Step3") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# 2. Konversi df_merged (Pandas) menjadi PySpark DataFrame
df_spark = spark.createDataFrame(df_merged)

df_filter = df_spark.filter(
    (F.col("kota") == "Magelang") & (F.col("total_harga") > 100000)
)
df_filter.show()

+------------+------------+----------+------+------------+-----------+--------------+--------+
|id_transaksi|id_pelanggan|  kategori|jumlah|harga_satuan|total_harga|nama_pelanggan|    kota|
+------------+------------+----------+------+------------+-----------+--------------+--------+
|         101|           1|Elektronik|     2|     1500000|    3000000|          Budi|Magelang|
|         103|           1|Elektronik|     1|     3000000|    3000000|          Budi|Magelang|
|         104|           3|   Makanan|    10|       15000|     150000|          Andi|Magelang|
+------------+------------+----------+------+------------+-----------+--------------+--------+



In [74]:
df_kategoripandas = df_merged.groupby("kategori")["total_harga"].sum()
df_kategoripandas

kategori
Elektronik    6000000
Makanan        310000
Pakaian        475000
Name: total_harga, dtype: int64

In [71]:
df_kategori = df_spark.groupBy("kategori").agg(F.sum("total_harga"))
df_kategori.show(5)

+----------+----------------+
|  kategori|sum(total_harga)|
+----------+----------------+
|Elektronik|         6000000|
|   Pakaian|          475000|
|   Makanan|          310000|
+----------+----------------+

